# Учебное исследование Ялуторовска с BlocksNet 1.0.0a9

Этот ноутбук шаг за шагом:

1. скачивает данные OpenStreetMap для Ялуторовска;
2. нарезает город на кварталы;
3. назначает землепользование;
4. **останавливается**, чтобы вы могли обсудить метрики;
5. считает доступность, население, школы, плотность и морфотипы;
6. сохраняет все данные и результаты в `examples/data/yalutorovsk`.

Ноутбук **ничего не запускает сам**: вы выполняете ячейки по порядку, нажимая `Shift+Enter`.

> 🛑 После подготовки кварталов будет остановка. Это специально: перед расчётами нужно решить, какие показатели вам нужны.

**Источник данных:** OpenStreetMap, © участники OpenStreetMap, лицензия ODbL.


## Как запустить

1. Откройте PowerShell в папке проекта:

   ```powershell
   cd "C:\Users\pc\Desktop\р\python-package-boilerplate"
   ```

2. Активируйте окружение:

   ```powershell
   .\.venv\Scripts\Activate.ps1
   ```

3. Запустите Jupyter:

   ```powershell
   .\.venv\Scripts\jupyter.exe lab
   ```

4. В браузере откройте `examples` → `yalutorovsk_study.ipynb`.

5. Ячейки запускайте сверху вниз. Первый запуск может занять 5–15 минут: данные скачиваются из OpenStreetMap.

**Если браузер не открылся**, скопируйте ссылку из терминала (обычно `http://localhost:8888/lab`).

**Проверка окружения:** первая ячейка кода печатает версию BlocksNet и путь к Python. Должно быть `1.0.0a9` и путь, оканчивающийся на `.venv\Scripts\python.exe`.


## Что означают термины

- **Квартал** — небольшой участок города, ограниченный дорогами, водой и железными дорогами.
- **Матрица доступности** — таблица, где для каждой пары кварталов записано время пути в минутах.
- **Обеспеченность** — какая доля потребности (например, в школах) закрывается в пределах заданного времени.
- **FSI** — отношение общей площади этажей к площади квартала. Показывает, насколько плотно застроен квартал.
- **GSI** — доля площади квартала, занятая зданиями.
- **MXI** — доля жилой площади в общей площади застройки.
- **Морфотип** — тип застройки: малоэтажная, среднеэтажная, многоэтажная, микрорайон и т.п.
- **Доступность территории** — средневзвешенное время, за которое из квартала можно добраться до остальных кварталов.


## 0. Проверка окружения

Запустите эту ячейку первой. Она ничего не скачивает.


In [ ]:
%matplotlib inline

import pickle
import sys
from importlib.metadata import version
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import osmnx as ox
import pandas as pd

plt.rcParams["figure.figsize"] = (10, 7)

print("Python:", sys.version.split()[0])
print("BlocksNet:", version("blocksnet"))
print("IduEdu:", version("iduedu"))
print("GeoPandas:", version("geopandas"))
print("OSMnx:", version("osmnx"))
print("Путь к Python:", sys.executable)


## 1. Настройки

Здесь собраны все параметры. Менять код ниже не нужно — достаточно поменять значения в этой ячейке.

- `CITY` — город и регион для поиска границы в OpenStreetMap.
- `USE_CACHE` — `True`: читать уже скачанные файлы; `False`: скачать данные заново.
- `STOP_AFTER_BLOCKS` — `True`: остановиться после кварталов; `False`: продолжить расчёты.
- `WALK_SPEED_KMH` — скорость пешехода. В BlocksNet/IduEdu по умолчанию 5 км/ч.


In [ ]:
# ===== ОСНОВНЫЕ НАСТРОЙКИ =====
CITY = "Ялуторовск, Тюменская область, Россия"

# True — читать готовые файлы; False — скачать данные заново
USE_CACHE = True

# 🛑 True — остановиться после подготовки кварталов.
# После обсуждения метрик поменяйте на False.
STOP_AFTER_BLOCKS = False

# Пешеходный граф: 5 км/ч
WALK_SPEED_KMH = 5.0
WALK_SPEED_M_PER_MIN = WALK_SPEED_KMH * 1000 / 60

# Нормативы обеспеченности школами берём из конфигурации BlocksNet
from blocksnet.config import service_types_config

SCHOOL_DEMAND_PER_1000 = int(service_types_config["school"]["demand"])
SCHOOL_ACCESSIBILITY_MIN = int(service_types_config["school"]["accessibility"])

# Папки: ноутбук можно запускать из корня проекта или из папки examples
PROJECT_DIR = Path.cwd()
EXAMPLES_DIR = PROJECT_DIR / "examples" if (PROJECT_DIR / "examples").is_dir() else PROJECT_DIR
DATA_DIR = EXAMPLES_DIR / "data" / "yalutorovsk"
RESULTS_DIR = DATA_DIR / "results"
DATA_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

paths = {
    "boundaries": DATA_DIR / "boundaries.geojson",
    "roads": DATA_DIR / "roads.geojson",
    "railways": DATA_DIR / "railways.geojson",
    "water": DATA_DIR / "water.geojson",
    "functional_zones": DATA_DIR / "functional_zones.geojson",
    "blocks": DATA_DIR / "blocks.geojson",
    "blocks_land_use": DATA_DIR / "blocks_land_use.geojson",
    "walk_graph": DATA_DIR / "walk_graph.pkl",
    "buildings": DATA_DIR / "buildings.geojson",
    "buildings_blocks": DATA_DIR / "buildings_blocks.geojson",
    "schools": DATA_DIR / "schools.geojson",
    "schools_blocks": DATA_DIR / "schools_blocks.geojson",
    "accessibility_matrix": RESULTS_DIR / "accessibility_matrix.csv",
    "provision": RESULTS_DIR / "provision.csv",
    "provision_links": RESULTS_DIR / "provision_links.csv",
    "density": RESULTS_DIR / "density.csv",
    "spacematrix": RESULTS_DIR / "spacematrix.csv",
    "spacematrix_clusters": RESULTS_DIR / "spacematrix_clusters.csv",
    "strelka": RESULTS_DIR / "strelka.csv",
    "area_accessibility": RESULTS_DIR / "area_accessibility.csv",
    "blocks_indicators": RESULTS_DIR / "blocks_indicators.geojson",
    "blocks_indicators_csv": RESULTS_DIR / "blocks_indicators.csv",
    "summary": RESULTS_DIR / "summary.csv",
}

print("Город:", CITY)
print("Данные:", DATA_DIR.resolve())
print("Результаты:", RESULTS_DIR.resolve())
print(
    "Норматив школ:",
    SCHOOL_DEMAND_PER_1000,
    "мест на 1000 жителей;",
    SCHOOL_ACCESSIBILITY_MIN,
    "мин",
)


## 2. Вспомогательные функции

Эти функции сохраняют и читают GeoJSON, а также скачивают слои из OpenStreetMap. Если файл уже есть и `USE_CACHE = True`, повторно скачивать не будем.


In [ ]:
def save_geojson(gdf, path):
    """Сохраняет слой в GeoJSON (в WGS84)."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    gdf = gdf.copy()
    gdf.to_crs(4326).to_file(path, driver="GeoJSON")
    return path


def load_geojson(path, target_crs):
    """Читает GeoJSON и переводит в нужную проекцию."""
    return gpd.read_file(path).to_crs(target_crs)


def fetch_osm(area, tags, geom_types, label):
    """Скачивает объекты OpenStreetMap и оставляет нужные типы геометрии."""
    try:
        gdf = ox.features_from_polygon(area, tags=tags)
    except Exception as exc:
        print(f"⚠️ {label}: данные OSM не найдены ({exc})")
        return gpd.GeoDataFrame(geometry=[], crs="EPSG:4326")

    gdf = gdf[gdf.geom_type.isin(geom_types)].copy()
    print(f"{label}: {len(gdf)} объектов")
    return gdf


def load_or_fetch(path, fetch_func, target_crs, label, keep_columns=None):
    """Читает готовый GeoJSON или скачивает слой и сохраняет его."""
    path = Path(path)

    if USE_CACHE and path.exists():
        print(f"[кэш] {label}: {path.name}")
        return load_geojson(path, target_crs)

    print(f"[OSM] {label}...")
    gdf = fetch_func()

    if gdf is None:
        gdf = gpd.GeoDataFrame(geometry=[], crs="EPSG:4326")

    if gdf.empty:
        print(f"⚠️ {label}: слой пустой")
        return gdf.to_crs(target_crs)

    # Для дорог/воды достаточно геометрии: атрибуты OSM бывают списками,
    # а GeoJSON не любит такие значения.
    if keep_columns is not None:
        cols = [c for c in keep_columns if c in gdf.columns]
        if "geometry" not in cols:
            cols.append("geometry")
        gdf = gdf[cols].copy()

    save_geojson(gdf, path)
    print(f"[OK] {label}: {len(gdf)} объектов -> {path.name}")
    return gdf.to_crs(target_crs)


def load_blocks(path, target_crs):
    """Читает кварталы и восстанавливает непрерывный block_id."""
    blocks = load_geojson(path, target_crs)

    if "block_id" not in blocks.columns:
        blocks["block_id"] = range(len(blocks))

    blocks["block_id"] = blocks["block_id"].astype(int)
    blocks = blocks.sort_values("block_id").reset_index(drop=True)
    blocks["block_id"] = range(len(blocks))
    return blocks


def save_blocks(blocks, path):
    """Сохраняет кварталы, превращая enum-значения в строки."""
    blocks = blocks.copy()
    if "block_id" not in blocks.columns:
        blocks["block_id"] = range(len(blocks))
    for col in blocks.columns:
        if blocks[col].dtype == "object":
            blocks[col] = blocks[col].apply(lambda v: v.value if hasattr(v, "value") else v)
    save_geojson(blocks, path)


## 3. Граница города

Границу берём из OpenStreetMap по названию города. Затем переводим её в метрическую проекцию UTM, чтобы площади и расстояния считались в метрах, а не в градусах.

**Результат:** `boundaries.geojson`, карта и площадь территории.


In [ ]:
if USE_CACHE and paths["boundaries"].exists():
    boundaries = load_geojson(paths["boundaries"], 4326)
    print("[кэш] boundaries.geojson")
else:
    print("[OSM] граница города...")
    boundaries = ox.geocode_to_gdf(CITY)[["geometry"]].copy()
    save_geojson(boundaries, paths["boundaries"])
    print("[OK] boundaries.geojson")

boundaries = boundaries.to_crs(4326)

if boundaries.empty:
    raise RuntimeError("Не удалось получить границу города. Проверьте CITY и интернет.")

area_wgs84 = boundaries.union_all()
crs = boundaries.estimate_utm_crs()
boundaries = boundaries.to_crs(crs)

print("Граница города")
print("  CRS:", crs)
print("  Площадь:", round(boundaries.area.sum() / 1_000_000, 2), "км²")
print("  Объектов:", len(boundaries))

ax = boundaries.plot(facecolor="#d9ead3", edgecolor="#38761d", linewidth=2)
ax.set_title(f"Граница: {CITY}")
ax.set_axis_off()
plt.show()


## 4. Дороги, железные дороги и вода

Эти слои нужны, чтобы нарезать город на кварталы. Скачиваем их из OpenStreetMap и сохраняем в GeoJSON. При повторном запуске файлы просто читаются с диска.

**Результат:** `roads.geojson`, `railways.geojson`, `water.geojson`.


In [ ]:
roads_tags = {
    "highway": [
        "construction", "crossing", "living_street", "motorway", "motorway_link",
        "motorway_junction", "pedestrian", "primary", "primary_link", "raceway",
        "residential", "road", "secondary", "secondary_link", "services",
        "tertiary", "tertiary_link", "track", "trunk", "trunk_link",
        "turning_circle", "turning_loop", "unclassified",
    ],
    "service": ["living_street", "emergency_access"],
}

railways_tags = {"railway": "rail"}

water_tags = {
    "riverbank": True,
    "reservoir": True,
    "basin": True,
    "dock": True,
    "canal": True,
    "pond": True,
    "natural": ["water", "bay"],
    "waterway": ["river", "canal", "ditch"],
    "landuse": "basin",
    "water": "lake",
}

roads = load_or_fetch(
    paths["roads"],
    lambda: fetch_osm(area_wgs84, roads_tags, ["LineString", "MultiLineString"], "дороги"),
    crs,
    "дороги",
    keep_columns=[],
)

railways = load_or_fetch(
    paths["railways"],
    lambda: fetch_osm(area_wgs84, railways_tags, ["LineString", "MultiLineString"], "железные дороги"),
    crs,
    "железные дороги",
    keep_columns=[],
)

water = load_or_fetch(
    paths["water"],
    lambda: fetch_osm(
        area_wgs84,
        water_tags,
        ["Polygon", "MultiPolygon", "LineString", "MultiLineString"],
        "вода",
    ),
    crs,
    "вода",
    keep_columns=[],
)

if roads.empty:
    raise RuntimeError("В OSM не найдены дороги. Без них нельзя нарезать кварталы.")

print("Дороги:", len(roads))
print("Железные дороги:", len(railways))
print("Вода:", len(water))


## 5. Нарезка кварталов

BlocksNet соединяет дороги, железные дороги и воду в барьеры и нарезает ими территорию города на замкнутые кварталы. Слишком маленькие или битые фрагменты убираются.

**Результат:** `blocks.geojson`, число кварталов, карта и средняя площадь.


In [ ]:
from blocksnet.blocks.cutting import preprocess_urban_objects, cut_urban_blocks
from blocksnet.blocks.postprocessing import postprocess_urban_blocks

if USE_CACHE and paths["blocks"].exists():
    blocks = load_blocks(paths["blocks"], crs)
    print("[кэш] blocks.geojson")
else:
    print("[BlocksNet] нарезка кварталов...")
    lines, polygons = preprocess_urban_objects(roads, railways, water)
    blocks = cut_urban_blocks(boundaries, lines, polygons)
    blocks = postprocess_urban_blocks(blocks)
    blocks = blocks.reset_index(drop=True)
    blocks["block_id"] = range(len(blocks))
    save_blocks(blocks, paths["blocks"])
    print("[OK] blocks.geojson")

if blocks.empty:
    raise RuntimeError("Кварталы не получились. Проверьте дороги и границу города.")

print("Кварталов:", len(blocks))
print("Средняя площадь квартала:", round(blocks.area.mean() / 10_000, 2), "га")
print("Суммарная площадь:", round(blocks.area.sum() / 1_000_000, 2), "км²")

ax = blocks.plot(facecolor="#fce5cd", edgecolor="#b45f06", linewidth=0.5)
ax.set_title(f"Кварталы: {len(blocks)}")
ax.set_axis_off()
plt.show()


## 6. Землепользование

Функциональные зоны OpenStreetMap (`landuse`) показывают, где жилые, деловые, промышленные, рекреационные и другие территории. BlocksNet сопоставляет их с кварталами и выбирает преобладающую категорию.

**Результат:** `functional_zones.geojson` и `blocks_land_use.geojson`.


In [ ]:
if USE_CACHE and paths["functional_zones"].exists():
    functional_zones = load_geojson(paths["functional_zones"], crs)
    print("[кэш] functional_zones.geojson")
else:
    functional_zones = fetch_osm(
        area_wgs84,
        {"landuse": True},
        ["Polygon", "MultiPolygon"],
        "землепользование",
    )

    if functional_zones.empty:
        print("⚠️ В OSM не найдены функциональные зоны.")
    elif "landuse" not in functional_zones.columns:
        print("⚠️ В данных OSM нет колонки landuse.")
        functional_zones = functional_zones.iloc[0:0]
    else:
        functional_zones = functional_zones.reset_index(drop=True)
        functional_zones = functional_zones[["geometry", "landuse"]].rename(
            columns={"landuse": "functional_zone"}
        )
        functional_zones["functional_zone"] = functional_zones["functional_zone"].apply(
            lambda v: v[0] if isinstance(v, list) and len(v) > 0 else v
        )
        functional_zones = functional_zones.dropna(subset=["functional_zone"]).copy()
        functional_zones["functional_zone"] = functional_zones["functional_zone"].astype(str)
        functional_zones = functional_zones[
            functional_zones.geometry.notna() & ~functional_zones.geometry.is_empty
        ].copy()
        save_geojson(functional_zones, paths["functional_zones"])
        print("[OK] functional_zones.geojson")

print("Функциональных зон:", len(functional_zones))
if not functional_zones.empty:
    print(functional_zones["functional_zone"].value_counts().head(10))


### Назначение землепользования кварталам

Правила ниже переводят теги OpenStreetMap в категории BlocksNet. Если тег не указан в правилах, квартал останется без категории — это нормально.

**Результат:** у каждого квартала появляется колонка `land_use` (там, где данных OSM хватило).


In [ ]:
from blocksnet.blocks.assignment import assign_land_use
from blocksnet.enums import LandUse

rules = {
    "commercial": LandUse.BUSINESS,
    "industrial": LandUse.INDUSTRIAL,
    "cemetery": LandUse.SPECIAL,
    "garages": LandUse.INDUSTRIAL,
    "residential": LandUse.RESIDENTIAL,
    "retail": LandUse.BUSINESS,
    "grass": LandUse.RECREATION,
    "farmland": LandUse.AGRICULTURE,
    "construction": LandUse.SPECIAL,
    "brownfield": LandUse.INDUSTRIAL,
    "forest": LandUse.RECREATION,
    "recreation_ground": LandUse.RECREATION,
    "religious": LandUse.SPECIAL,
    "flowerbed": LandUse.RECREATION,
    "military": LandUse.SPECIAL,
    "landfill": LandUse.TRANSPORT,
}

if USE_CACHE and paths["blocks_land_use"].exists():
    blocks = load_blocks(paths["blocks_land_use"], crs)
    print("[кэш] blocks_land_use.geojson")
else:
    if functional_zones.empty:
        zones = functional_zones
    else:
        zones = functional_zones[functional_zones["functional_zone"].isin(rules)].copy()

    if zones.empty:
        blocks["land_use"] = None
        print("⚠️ Нет подходящих функциональных зон — колонка land_use пустая.")
    else:
        blocks = assign_land_use(blocks, zones, rules)
        # assign_land_use оставляет только геометрию и доли землепользования,
        # поэтому восстанавливаем block_id и непрерывную нумерацию.
        blocks = blocks.reset_index(drop=True)
        blocks["block_id"] = range(len(blocks))
        blocks["land_use"] = blocks["land_use"].apply(
            lambda v: v.value if hasattr(v, "value") else v
        )
        print("[OK] землепользование назначено")

    save_blocks(blocks, paths["blocks_land_use"])

print(blocks["land_use"].value_counts(dropna=False))

ax = blocks.plot(color="#eeeeee", edgecolor="white", linewidth=0.3)
if blocks["land_use"].notna().any():
    blocks.plot(
        column="land_use",
        legend=True,
        ax=ax,
        categorical=True,
        cmap="tab10",
    )
ax.set_title("Землепользование кварталов")
ax.set_axis_off()
plt.show()


# 🛑 ОСТАНОВКА: кварталы готовы

**Дальше не запускайте ячейки, пока мы не обсудим метрики.**

Сейчас готово:

- граница Ялуторовска;
- дороги, железные дороги, вода;
- кварталы (`blocks.geojson`);
- землепользование (`blocks_land_use.geojson`).

Ещё **не** рассчитано: население, школы, доступность, плотность, морфотипы.

## Какие метрики доступны

По умолчанию ноутбук посчитает всё, что есть в `pipeline.ipynb` BlocksNet 1.0.0a9:

1. **Матрица доступности** — время пешком между кварталами, в минутах.
2. **Обеспеченность школами** — `provision_strong`: доля спроса, покрытая в пределах 15 минут.
3. **Плотностные показатели** — FSI, GSI, MXI, L, OSR, доли жилой и нежилой площади.
4. **Морфотипы** — SpaceMatrix (кластеры) и Strelka.
5. **Доступность территории** — средневзвешенное время до всех кварталов.

Дополнительно BlocksNet умеет считать обеспеченность детсадами, поликлиниками, магазинами и другими сервисами: в конфигурации более 80 типов. Если нужны — скажите, добавим.

**Напишите в чат, какие метрики нужны. После этого поменяйте `STOP_AFTER_BLOCKS` на `False` в ячейке настроек и запустите ячейки ниже.**


In [ ]:
if STOP_AFTER_BLOCKS:
    raise RuntimeError(
        "🛑 Остановка после подготовки кварталов.\n\n"
        "Сейчас расчёты не запускаются. Напишите в чат, какие метрики нужны.\n"
        "Чтобы продолжить: в ячейке настроек поменяйте STOP_AFTER_BLOCKS = False "
        "и запустите ячейки ниже."
    )

print("✅ Остановка отключена — можно продолжать расчёты.")


## 7. Пешеходный граф

**Источник:** OpenStreetMap (загружается через IduEdu, которую использует BlocksNet).
**Способ передвижения:** пешком.
**Скорость:** 5 км/ч (83,33 м/мин).
**Время:** в минутах, атрибут рёбер `time_min`.

Интермодальный граф не строим. Граф скачивается один раз и сохраняется в `walk_graph.pkl`.

**Результат:** граф улиц и число узлов/рёбер.


In [ ]:
from blocksnet.relations import get_accessibility_graph

if USE_CACHE and paths["walk_graph"].exists():
    with open(paths["walk_graph"], "rb") as f:
        graph = pickle.load(f)
    print("[кэш] walk_graph.pkl")
else:
    print("[OSM] строю пешеходный граф (5 км/ч)...")
    graph = get_accessibility_graph(
        boundaries,
        "walk",
        walk_speed=WALK_SPEED_M_PER_MIN,
    )
    with open(paths["walk_graph"], "wb") as f:
        pickle.dump(graph, f, protocol=pickle.HIGHEST_PROTOCOL)
    print("[OK] walk_graph.pkl")

print("Узлов:", graph.number_of_nodes())
print("Рёбер:", graph.number_of_edges())
print("CRS графа:", graph.graph.get("crs"))
print("Тип:", graph.graph.get("type"))
print("Скорость, м/мин:", round(float(graph.graph.get("walk_speed", WALK_SPEED_M_PER_MIN)), 2))

edge_times = [
    data["time_min"]
    for _, _, data in graph.edges(data=True)
    if "time_min" in data
]
if edge_times:
    print("Медианное время ребра:", round(float(np.median(edge_times)), 2), "мин")


## 8. Матрица доступности

Считаем кратчайшее время **пешком** между центрами всех кварталов. Получается таблица N×N, где N — число кварталов. Значения в минутах; `inf` означает, что путь не найден.

**Результат:** `results/accessibility_matrix.csv`, статистика и гистограмма.


In [ ]:
from blocksnet.relations import calculate_accessibility_matrix

if USE_CACHE and paths["accessibility_matrix"].exists():
    acc_mx = pd.read_csv(paths["accessibility_matrix"], index_col=0)
    acc_mx.index = acc_mx.index.astype(int)
    acc_mx.columns = acc_mx.columns.astype(int)
    print("[кэш] accessibility_matrix.csv")
else:
    print("[BlocksNet] расчёт матрицы доступности...")
    acc_mx = calculate_accessibility_matrix(blocks, graph, weight_key="time_min")
    acc_mx.to_csv(paths["accessibility_matrix"])
    print("[OK] accessibility_matrix.csv")

if set(acc_mx.index) != set(blocks.index):
    raise RuntimeError("Матрица не совпадает с кварталами. Удалите кэш и запустите заново.")

values = acc_mx.to_numpy(dtype=float)
finite = values[np.isfinite(values)]

print("Размер матрицы:", acc_mx.shape)

if finite.size:
    print("Минимум:", round(float(finite.min()), 2), "мин")
    print("Медиана:", round(float(np.median(finite)), 2), "мин")
    print("Максимум:", round(float(finite.max()), 2), "мин")
    print("Недостижимых пар:", round(float(np.isinf(values).mean() * 100), 2), "%")

    plt.hist(finite, bins=40, color="#6aa84f", edgecolor="white")
    plt.title("Распределение времени в пути между кварталами")
    plt.xlabel("Минуты")
    plt.ylabel("Число пар кварталов")
    plt.show()
else:
    print("⚠️ Все пары кварталов недостижимы. Проверьте граф.")


## 9. Здания и население

Загружаем здания из OpenStreetMap. Число этажей берём из тега `building:levels`; если тега нет — считаем 1 этаж. Жилая площадь — 90% общей, население — оценка: жилая площадь / 30 м² на человека.

> Это **оценка**, а не официальная статистика. Для точных выводов нужны данные администрации.

**Результат:** `buildings.geojson`, `buildings_blocks.geojson`, карта населения.


In [ ]:
from blocksnet.blocks.aggregation import aggregate_objects
from blocksnet.preprocessing.imputing import impute_buildings

if USE_CACHE and paths["buildings_blocks"].exists():
    buildings_blocks = load_geojson(paths["buildings_blocks"], crs)
    if paths["buildings"].exists():
        buildings = load_geojson(paths["buildings"], crs)
    else:
        buildings = gpd.GeoDataFrame()
    print("[кэш] buildings_blocks.geojson")
else:
    if USE_CACHE and paths["buildings"].exists():
        buildings = load_geojson(paths["buildings"], crs)
        print("[кэш] buildings.geojson")
    else:
        buildings = fetch_osm(
            area_wgs84,
            {"building": True},
            ["Polygon", "MultiPolygon"],
            "здания",
        )
        if buildings.empty:
            raise RuntimeError("В OSM не найдены здания. Дальнейшие показатели без них не посчитать.")

        buildings = buildings[
            buildings.geom_type.isin(["Polygon", "MultiPolygon"])
        ].copy()
        # OSMnx возвращает MultiIndex (element, id). BlocksNet ждёт обычный индекс.
        buildings = buildings.reset_index(drop=True)

        is_living_tags = [
            "residential", "house", "apartments", "detached", "terrace", "dormitory",
        ]

        def is_living(value):
            values = value if isinstance(value, list) else [value]
            return any(v in is_living_tags for v in values)

        buildings["is_living"] = buildings["building"].apply(is_living)

        def parse_levels(value):
            """Достаёт число этажей из тега OSM, даже если там список."""
            if isinstance(value, (list, tuple)):
                value = value[0] if value else None
            if value is None:
                return np.nan
            text = str(value).replace(",", ".").split(";")[0].strip()
            try:
                return float(text)
            except ValueError:
                return np.nan

        if "building:levels" in buildings.columns:
            buildings["number_of_floors"] = buildings["building:levels"].apply(parse_levels)
        else:
            buildings["number_of_floors"] = np.nan

        buildings = impute_buildings(buildings, default_living_demand=30)

        # Оставляем только нужные поля: в OSM бывают списки в атрибутах,
        # а GeoJSON такие значения не поддерживает.
        building_cols = [
            "geometry",
            "is_living",
            "number_of_floors",
            "footprint_area",
            "build_floor_area",
            "living_area",
            "non_living_area",
            "population",
        ]
        buildings = buildings[[c for c in building_cols if c in buildings.columns]].copy()
        save_geojson(buildings, paths["buildings"])
        print("[OK] buildings.geojson")

    buildings_blocks, excess_buildings = aggregate_objects(blocks, buildings)
    buildings_blocks["site_area"] = buildings_blocks.area
    save_geojson(buildings_blocks, paths["buildings_blocks"])
    print("[OK] buildings_blocks.geojson")
    print("Зданий вне кварталов:", len(excess_buildings))

print("Зданий:", len(buildings))
print("Население (оценка):", int(buildings_blocks["population"].sum()), "чел.")

ax = buildings_blocks.plot(
    column="population",
    cmap="viridis",
    legend=True,
    edgecolor="white",
    linewidth=0.3,
)
ax.set_title("Население по кварталам (оценка)")
ax.set_axis_off()
plt.show()


## 10. Школы

Загружаем школы из OpenStreetMap (`amenity=school`). Если вместимость не указана, BlocksNet подставляет минимальную вместимость из своего справочника (250 мест). Затем суммируем вместимость по кварталам.

**Результат:** `schools.geojson`, `schools_blocks.geojson`, карта вместимости.


In [ ]:
from blocksnet.preprocessing.imputing import impute_services

if USE_CACHE and paths["schools_blocks"].exists():
    schools_blocks = load_geojson(paths["schools_blocks"], crs)
    if paths["schools"].exists():
        schools = load_geojson(paths["schools"], crs)
    else:
        schools = gpd.GeoDataFrame()
    print("[кэш] schools_blocks.geojson")
else:
    if USE_CACHE and paths["schools"].exists():
        schools = load_geojson(paths["schools"], crs)
        print("[кэш] schools.geojson")
    else:
        schools = fetch_osm(
            area_wgs84,
            {"amenity": "school"},
            ["Polygon", "MultiPolygon", "Point"],
            "школы",
        )
        # OSMnx возвращает MultiIndex (element, id). BlocksNet ждёт обычный индекс.
        schools = schools.reset_index(drop=True)
        if not schools.empty:
            schools = impute_services(schools, "school")
            schools = schools[["geometry", "capacity"]].copy()
            save_geojson(schools, paths["schools"])
            print("[OK] schools.geojson")

    if schools.empty:
        print("⚠️ Школы в OSM не найдены — обеспеченность не будет рассчитана.")
        schools_blocks = blocks[["geometry"]].copy()
        schools_blocks["capacity"] = 0.0
    else:
        schools_blocks, excess_schools = aggregate_objects(blocks, schools)
        if "capacity" not in schools_blocks.columns:
            schools_blocks["capacity"] = 0.0
        schools_blocks["capacity"] = schools_blocks["capacity"].fillna(0)
        print("Школ вне кварталов:", len(excess_schools))

    save_geojson(schools_blocks, paths["schools_blocks"])

print("Школ (объектов):", len(schools))
print("Вместимость школ:", int(schools_blocks["capacity"].sum()), "мест")

ax = schools_blocks.plot(
    column="capacity",
    cmap="OrRd",
    legend=True,
    edgecolor="white",
    linewidth=0.3,
)
ax.set_title("Вместимость школ по кварталам")
ax.set_axis_off()
plt.show()


## 11. Обеспеченность школами

BlocksNet считает, какая доля потребности в школах закрывается в пределах **15 минут** пешком. Норматив — **120 мест на 1000 жителей** (из конфигурации BlocksNet).

- `provision_strong` — доля спроса, покрытая в пределах 15 минут.
- `provision_weak` — вспомогательный показатель с менее строгим ограничением.

**Результат:** `results/provision.csv`, `results/provision_links.csv`, карта.


In [ ]:
from blocksnet.analysis.provision import competitive_provision, provision_strong_total

df = buildings_blocks[["population"]].join(schools_blocks[["capacity"]])
df["population"] = df["population"].fillna(0)
df["capacity"] = df["capacity"].fillna(0)

if df["capacity"].sum() > 0:
    prov_df, links_df = competitive_provision(
        df,
        acc_mx,
        SCHOOL_ACCESSIBILITY_MIN,
        SCHOOL_DEMAND_PER_1000,
    )
    prov_df.to_csv(paths["provision"])
    links_df.to_csv(paths["provision_links"])

    print("Сильная обеспеченность (всего):", round(provision_strong_total(prov_df), 3))
    print("Средняя по кварталам:", round(float(prov_df["provision_strong"].mean()), 3))
    print("Связей школа-квартал:", len(links_df))

    ax = blocks.plot(color="#eeeeee", edgecolor="white", linewidth=0.3)
    blocks.join(prov_df[["provision_strong"]]).plot(
        column="provision_strong",
        cmap="RdYlGn",
        vmin=0,
        vmax=1,
        legend=True,
        ax=ax,
    )
    ax.set_title("Обеспеченность школами (доля спроса в пределах 15 минут)")
    ax.set_axis_off()
    plt.show()
else:
    print("⚠️ Нет данных о вместимости школ — обеспеченность не рассчитана.")
    prov_df = pd.DataFrame(index=blocks.index)
    prov_df["provision_strong"] = np.nan
    prov_df["provision_weak"] = np.nan
    links_df = pd.DataFrame(columns=["source", "target", "value"])


## 12. Плотностные показатели

Считаем по каждому кварталу:

- **FSI** — сколько квадратных метров этажей приходится на квадратный метр территории;
- **GSI** — какая доля территории занята зданиями;
- **MXI** — доля жилой площади;
- **L** — средняя этажность;
- **OSR** — доля открытых пространств;
- **share_living / share_non_living** — доли жилой и нежилой площади.

**Результат:** `results/density.csv` и карты.


In [ ]:
from blocksnet.analysis.indicators import calculate_density_indicators

density_df = calculate_density_indicators(buildings_blocks)

density_cols = [
    "fsi",
    "gsi",
    "mxi",
    "l",
    "osr",
    "share_living",
    "share_non_living",
]

density_df[density_cols].to_csv(paths["density"])
print(density_df[density_cols].describe().T[["mean", "min", "max"]].round(3))

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for ax, col in zip(axes.flat, density_cols):
    blocks.join(density_df[[col]]).plot(
        column=col,
        cmap="viridis",
        legend=True,
        ax=ax,
        edgecolor="white",
        linewidth=0.2,
    )
    ax.set_title(col)
    ax.set_axis_off()

axes.flat[-1].axis("off")
fig.suptitle("Плотностные показатели по кварталам", fontsize=14)
plt.tight_layout()
plt.show()


## 13. Морфотипы SpaceMatrix

SpaceMatrix группирует застроенные кварталы по плотности и этажности и присваивает им понятные названия: малоэтажная, среднеэтажная, многоэтажная застройка и т.п.

**Результат:** `results/spacematrix.csv`, `results/spacematrix_clusters.csv`, карта.


In [ ]:
from blocksnet.analysis.morphotypes import get_spacematrix_morphotypes

n_developed = int((density_df["fsi"] > 0).sum())

if n_developed >= 2:
    n_clusters = min(11, n_developed)
    spacematrix_df, spacematrix_clusters = get_spacematrix_morphotypes(
        density_df,
        n_clusters=n_clusters,
    )
    spacematrix_df[["cluster", "morphotype"]].to_csv(paths["spacematrix"])
    spacematrix_clusters.to_csv(paths["spacematrix_clusters"], index_label="cluster")

    print("Кластеров:", n_clusters)
    print(spacematrix_df["morphotype"].value_counts())

    ax = blocks.plot(color="#eeeeee", edgecolor="white", linewidth=0.3)
    blocks.join(spacematrix_df[["morphotype"]]).plot(
        column="morphotype",
        legend=True,
        ax=ax,
        categorical=True,
        cmap="tab20",
    )
    ax.set_title("Морфотипы SpaceMatrix")
    ax.set_axis_off()
    plt.show()
else:
    print("⚠️ Слишком мало застроенных кварталов для SpaceMatrix.")
    spacematrix_df = pd.DataFrame(index=density_df.index)
    spacematrix_df["morphotype"] = None
    spacematrix_clusters = pd.DataFrame()


## 14. Морфотипы Strelka

Strelka — второй способ классификации застройки. Он смотрит на этажность, плотность и долю жилья и присваивает кварталу один из готовых типов.

**Результат:** `results/strelka.csv` и карта.


In [ ]:
from blocksnet.analysis.morphotypes import get_strelka_morphotypes

strelka_df = get_strelka_morphotypes(density_df)
strelka_df[["morphotype"]].to_csv(paths["strelka"])

print(strelka_df["morphotype"].value_counts(dropna=False))

if strelka_df["morphotype"].notna().any():
    ax = blocks.plot(color="#eeeeee", edgecolor="white", linewidth=0.3)
    blocks.join(strelka_df[["morphotype"]]).plot(
        column="morphotype",
        legend=True,
        ax=ax,
        categorical=True,
        cmap="tab20",
    )
    ax.set_title("Морфотипы Strelka")
    ax.set_axis_off()
    plt.show()
else:
    print("⚠️ Морфотипы Strelka не определены: нет застроенных кварталов.")


## 15. Доступность территории

Для каждого квартала считаем средневзвешенное время до всех остальных кварталов. Вес — площадь квартала. Значение `inf` означает, что часть кварталов недостижима.

**Результат:** `results/area_accessibility.csv` и карта.


In [ ]:
from blocksnet.analysis.network.accessibility import area_accessibility

area_acc = area_accessibility(acc_mx, buildings_blocks)
area_acc.to_csv(paths["area_accessibility"])

print(
    "Средняя доступность территории:",
    round(float(area_acc["area_accessibility"].replace(np.inf, np.nan).mean()), 2),
    "мин",
)
print(
    "Кварталов без пути:",
    int(np.isinf(area_acc["area_accessibility"]).sum()),
)

area_plot = area_acc.replace([np.inf, -np.inf], np.nan)

ax = blocks.plot(color="#eeeeee", edgecolor="white", linewidth=0.3)
blocks.join(area_plot).plot(
    column="area_accessibility",
    cmap="RdYlGn_r",
    legend=True,
    ax=ax,
)
ax.set_title("Доступность территории (минуты)")
ax.set_axis_off()
plt.show()


## 16. Сводка

Собираем все показатели в один слой `blocks_indicators.geojson` и короткую таблицу `summary.csv`. Эти файлы понадобятся для будущей HTML-страницы с картой и дашбордом.

**Результат:** `results/blocks_indicators.geojson`, `results/blocks_indicators.csv`, `results/summary.csv`.


In [ ]:
# Собираем все показатели в один слой
if "block_id" not in blocks.columns:
    blocks = blocks.reset_index(drop=True)
    blocks["block_id"] = range(len(blocks))

indicators = blocks[["block_id", "land_use", "geometry"]].copy()
indicators["site_area"] = blocks.area

indicators = indicators.join(
    buildings_blocks[
        [
            "population",
            "footprint_area",
            "build_floor_area",
            "living_area",
            "non_living_area",
        ]
    ]
)

indicators = indicators.join(
    schools_blocks[["capacity"]].rename(columns={"capacity": "school_capacity"})
)

if "provision_strong" in prov_df.columns:
    indicators = indicators.join(prov_df[["provision_strong", "provision_weak"]])

indicators = indicators.join(density_df[density_cols])

if "morphotype" in spacematrix_df.columns:
    indicators = indicators.join(
        spacematrix_df[["morphotype"]].rename(
            columns={"morphotype": "spacematrix_morphotype"}
        )
    )

if "morphotype" in strelka_df.columns:
    indicators = indicators.join(
        strelka_df[["morphotype"]].rename(
            columns={"morphotype": "strelka_morphotype"}
        )
    )

indicators = indicators.join(area_acc[["area_accessibility"]])

# GeoJSON не поддерживает бесконечности: заменяем их на пустые значения
indicators_geojson = indicators.copy()
for col in indicators_geojson.select_dtypes(include=[np.number]).columns:
    indicators_geojson[col] = indicators_geojson[col].replace([np.inf, -np.inf], np.nan)

save_geojson(indicators_geojson, paths["blocks_indicators"])
indicators.drop(columns="geometry").to_csv(
    paths["blocks_indicators_csv"],
    index=False,
    encoding="utf-8-sig",
)


def safe_mean(series):
    series = pd.to_numeric(series, errors="coerce").replace([np.inf, -np.inf], np.nan)
    return float(series.mean()) if series.notna().any() else float("nan")


prov_total = (
    provision_strong_total(prov_df)
    if "demand" in prov_df.columns and "provision_strong" in prov_df.columns
    else float("nan")
)
prov_mean = (
    safe_mean(prov_df["provision_strong"])
    if "provision_strong" in prov_df.columns
    else float("nan")
)

summary = pd.DataFrame(
    [
        ("Кварталов", len(blocks), "шт."),
        ("Площадь территории", round(blocks.area.sum() / 1_000_000, 2), "км²"),
        ("Население (оценка)", int(buildings_blocks["population"].sum()), "чел."),
        ("Вместимость школ", int(schools_blocks["capacity"].sum()), "мест"),
        ("Обеспеченность школами (всего)", round(prov_total, 3), "доля"),
        ("Средняя обеспеченность школами", round(prov_mean, 3), "доля"),
        ("Средний FSI", round(safe_mean(density_df["fsi"]), 3), ""),
        ("Средний GSI", round(safe_mean(density_df["gsi"]), 3), ""),
        ("Средняя доступность территории", round(safe_mean(area_acc["area_accessibility"]), 2), "мин"),
        (
            "Недостижимых пар",
            round(float(np.isinf(acc_mx.to_numpy(dtype=float)).mean() * 100), 2),
            "%",
        ),
    ],
    columns=["Показатель", "Значение", "Единица"],
)

summary.to_csv(paths["summary"], index=False, encoding="utf-8-sig")
print(summary.to_string(index=False))


## 17. Проверка результатов

Запустите эту ячейку после расчётов. Она проверит, что все файлы на месте, матрица имеет размер N×N, а индексы совпадают с кварталами.

Если увидите ❌ — прочитайте сообщение выше: обычно помогает удалить папку `examples/data/yalutorovsk` и запустить ноутбук заново.


In [ ]:
print("=== ПРОВЕРКА ФАЙЛОВ ===")

required = [
    paths["boundaries"],
    paths["roads"],
    paths["blocks"],
    paths["blocks_land_use"],
    paths["walk_graph"],
    paths["accessibility_matrix"],
    paths["buildings_blocks"],
    paths["density"],
    paths["area_accessibility"],
    paths["blocks_indicators"],
    paths["summary"],
]

optional = [
    paths["railways"],
    paths["water"],
    paths["functional_zones"],
    paths["schools"],
    paths["schools_blocks"],
    paths["provision"],
    paths["spacematrix"],
    paths["spacematrix_clusters"],
    paths["strelka"],
]

for path in required:
    print(("✅" if Path(path).exists() else "❌"), Path(path).relative_to(EXAMPLES_DIR))

for path in optional:
    print(("✅" if Path(path).exists() else "⚠️"), Path(path).relative_to(EXAMPLES_DIR))

missing = [path for path in required if not Path(path).exists()]

if missing:
    print("\n⚠️ Расчёты ещё не завершены: не хватает файлов выше.")
    print("Сначала выполните все ячейки до этой проверки.")
else:
    print("\n=== ПРОВЕРКА ДАННЫХ ===")
    print("Кварталов:", len(blocks))
    print("Матрица:", acc_mx.shape)

    assert acc_mx.shape == (len(blocks), len(blocks)), "Матрица должна быть N×N"
    assert set(acc_mx.index) == set(blocks.index), "Индексы матрицы и кварталов не совпадают"
    assert len(indicators) == len(blocks), "В сводке не все кварталы"
    print("✅ Матрица N×N, индексы совпадают, кварталы на месте")

    key_cols = [
        col
        for col in [
            "population",
            "school_capacity",
            "provision_strong",
            "fsi",
            "gsi",
            "area_accessibility",
        ]
        if col in indicators.columns
    ]

    print("\nПропуски в ключевых колонках:")
    print(indicators[key_cols].isna().sum())

    print("\nДиапазоны:")
    print(indicators[["fsi", "gsi", "mxi", "area_accessibility"]].describe().T)


## Повторный запуск и обслуживание

### Обычный повторный запуск

`Kernel → Restart & Run All`. Готовые файлы читаются из `examples/data/yalutorovsk`, поэтому повторный запуск быстрее.

Если `STOP_AFTER_BLOCKS = True`, полный прогон снова остановится после кварталов — это нормально.

### Скачать данные заново

1. В ячейке настроек поставьте `USE_CACHE = False`.
2. Запустите ноутбук заново.

Либо удалите папку `examples/data/yalutorovsk` — тогда все данные скачаются с нуля.

### Продолжить после остановки

1. В ячейке настроек поставьте `STOP_AFTER_BLOCKS = False`.
2. Запустите ячейку остановки и все ячейки ниже.

### Если что-то пошло не так

- **Нет интернета или OSM не отвечает** — повторите позже.
- **Матрица не совпадает с кварталами** — удалите кэш и запустите заново.
- **Предупреждение `torch.jit.script is deprecated`** — это не ошибка, а предупреждение библиотеки.
- **Меняли `CITY`** — удалите папку данных, чтобы не смешивать города.

### Что дальше

После проверки результатов отдельным этапом сделаем Python-файл, который читает сохранённые GeoJSON/CSV и собирает **одну HTML-страницу** с интерактивной картой, графиками, таблицами и выводами.

**Источник данных:** OpenStreetMap, © участники OpenStreetMap, лицензия ODbL.
